# Task 06 -- Joint multi-channel Tucker vs. downstream joint model (`docs/open_questions.md` §3.4, §3.5)

Runs `correlated_variables.md` §6's experiment exactly as specified there:
one partition (`wsh`/`wsd`/`wsp`), channels `{u, v, log E, F}` -- **`F`, not
`T`**, per §3.5's decorrelated-basis fix -- compared as (a) one joint
4-channel Tucker vs. (b) four independent per-channel Tuckers + CCA on the
coefficients, both against a shuffled null baseline. See
`tasks/06-joint-channel-tucker-vs-downstream.md`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorly as tl
import xarray as xr
from tensorly.decomposition import partial_tucker

from train.ingest import load_training_window

tl.set_backend("numpy")

SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("../data")

# Window entirely inside era 3 (starts 2025-10-27, core.archive_eras.ARCHIVE_ERAS);
# load_training_window rejects any window spanning more than one era.
T_START = datetime(2025, 11, 1, tzinfo=timezone.utc)
T_END = datetime(2026, 1, 1, tzinfo=timezone.utc)

from train.directional import ComingFromToUV

X_raw = load_training_window(DATA_DIR, T_START, T_END, ["wsh", "wsd", "wsp"])
X_uv = ComingFromToUV(pairs=[("wsh", "wsd", "u_wsea", "v_wsea")]).fit_transform(X_raw)

# variable_taxonomy.md §3.4: log E := log(H^2) = 2*log(H); T := log(T_m01).
# correlated_variables.md §6 / open_questions.md §3.5: F := log E - T,
# decorrelated from log E (depends only on m1, not m0) -- use F, not T, here.
log_E = 2.0 * np.log(X_raw["wsh"])
T_period = np.log(X_raw["wsp"])
F = log_E - T_period

def to_lat_lon_time(da):
    return da.transpose("wave_lat", "wave_lon", "time").values

channels = {"u": X_uv["u_wsea"], "v": X_uv["v_wsea"], "logE": log_E, "F": F}
arrays = [to_lat_lon_time(channels[c]) for c in ("u", "v", "logE", "F")]

ocean_mask = np.ones(arrays[0].shape[:2], dtype=bool)
for a in arrays:
    ocean_mask &= ~np.isnan(a).any(axis=-1)
filled = [np.where(ocean_mask[:, :, None], a, 0.0) for a in arrays]
stacked = np.stack(filled, axis=-1)  # (n_lat, n_lon, T, 4) -- the joint 4-channel tensor
tucker_mask = np.broadcast_to(ocean_mask[:, :, None, None], stacked.shape).astype(np.float64)
print(f"stacked shape {stacked.shape}, {ocean_mask.mean():.1%} ocean")


## Fit (a): one joint 4-channel Tucker

In [ ]:
R_LAT, R_LON, R_TIME, R_C = 8, 16, stacked.shape[2], 3
core_joint, factors_joint = partial_tucker(
    stacked, modes=[0, 1, 2, 3], rank=[R_LAT, R_LON, R_TIME, R_C],
    init="random", tol=1e-5, n_iter_max=2000, random_state=SEED, mask=tucker_mask,
)
A_C = factors_joint[3]  # (4, R_C) channel-mode factor -- inspect for a logE/F-coupled column
print("A_C (rows = u,v,logE,F):\n", A_C)


## Fit (b): four independent per-channel Tuckers, comparable rank budget

In [ ]:
independent_cores = {}
for name, arr in zip(("u", "v", "logE", "F"), filled):
    mask3 = tucker_mask[..., 0]
    c, _ = partial_tucker(
        arr, modes=[0, 1, 2], rank=[R_LAT, R_LON, R_TIME],
        init="random", tol=1e-5, n_iter_max=2000, random_state=SEED, mask=mask3,
    )
    independent_cores[name] = c  # (R_LAT, R_LON, T)


## Diagnostic, with a null baseline (`correlated_variables.md` §6, steps 4-5)

Compare `A_C`'s `logE`/`F` coupling strength against CCA on the vectorized
`independent_cores["logE"]` vs. `independent_cores["F"]` coefficient series.
Run **both** against a shuffled/synthetic null (permute the time axis of one
channel independently) before attributing any coupling to real physics
rather than residual structure. Record the comparison and the decision (per
§6's decision rule) in `tasks/06-joint-channel-tucker-vs-downstream.md` §4.

In [ ]:
from sklearn.cross_decomposition import CCA

logE_coef = independent_cores["logE"].reshape(-1, independent_cores["logE"].shape[-1]).T  # (T, R_LAT*R_LON)
F_coef = independent_cores["F"].reshape(-1, independent_cores["F"].shape[-1]).T

cca = CCA(n_components=1)
cca.fit(logE_coef, F_coef)
logE_c, F_c = cca.transform(logE_coef, F_coef)
real_corr = np.corrcoef(logE_c[:, 0], F_c[:, 0])[0, 1]

# Null baseline: shuffle F's time axis, refit CCA, compare.
F_shuffled = F_coef[rng.permutation(F_coef.shape[0])]
cca_null = CCA(n_components=1).fit(logE_coef, F_shuffled)
logE_n, F_n = cca_null.transform(logE_coef, F_shuffled)
null_corr = np.corrcoef(logE_n[:, 0], F_n[:, 0])[0, 1]

print(f"real logE/F canonical correlation: {real_corr:.3f}")
print(f"shuffled-null canonical correlation: {null_corr:.3f}")
